# Canadian Housing Affordability Data Cleaning

This notebook loads **Canada Mortgage and Housing Corporation (CMHC) Rental Market Survey (RMS)** data, 2020–2024, cleans **Table 4.1**, and prepares it for **exploratory data analysis (EDA)** and merging with **crime statistics (CSI)**.

## Step 1: Import Libraries
I will use pandas for dataframes and numpy for numerical operations.

In [1]:
import pandas as pd # for loading and manipulating .xlsx and .csv files
import numpy as np  # for calculations and handling NaN values

## Step 2: Define File Paths
I created a list of CMHC Excel files for years 2020–2024.

In [2]:
years = range(2020, 2025)
file_paths = [f"../data/raw/cmhc_rms_canada_{year}.xlsx" for year in years]

## Step 3: Read and clean apartment/condominium data
- Concatenate all 5 data sources into a single raw data frame
- Flatten and strip headers, then check updated format
- Create clean data frame: new column headers, no unneccessary columns, no rows for centre data that are null, metadata or annotations, or a "Total" calculation
- Format numeric column data

### Part 1:
- Concatenate all 5 data sources into a single raw data frame 

In [3]:
cmhc_list = [] # create a list to combine survey data from each year

for year, file in zip(years, file_paths):
    # focus on condo apartment sheet at header on row 7
    df = pd.read_excel(file, sheet_name='Table 4.1', header=6)
    # add a year col
    df['year'] = year
    # append each year together
    cmhc_list.append(df)

# create a new df with resequenced index
cmhc_raw = pd.concat(cmhc_list, ignore_index=True)
# print first 5 values of raw data
print(cmhc_raw.head())

                            Centre Rental \nCondo \nApts Unnamed: 2  \
0                      Halifax CMA                   0.9          a   
1                     Montréal CMA                   2.0          b   
2  Ottawa-Gatineau CMA (Qué. part)                    **        NaN   
3                       Québec CMA                   3.2          d   
4                     Hamilton CMA                   0.3          a   

   Apts in the \nRMS Unnamed: 4 Unnamed: 5 Rental \nCondo \nApts.1 Unnamed: 7  \
0                1.9          a          Δ                   1,666          c   
1                2.7          a          Δ                   1,277          a   
2                1.6          a        NaN                   1,206          b   
3                2.7          a          -                   1,093          a   
4                3.5          a          Δ                   1,947          c   

  Apts in the \nRMS.1 Unnamed: 9 Unnamed: 10 Rental \nCondo \nApts.2  \
0             

### Part 2:
- Flatten and strip headers, then check updated format

In [4]:
# Flatten headers and remove \n
cmhc_raw.columns = [col.replace('\n', ' ').strip() for col in cmhc_raw.columns]

# Inspect column names
print(cmhc_raw.columns.tolist())

# Check first 5 rows
print(cmhc_raw.head())

['Centre', 'Rental  Condo  Apts', 'Unnamed: 2', 'Apts in the  RMS', 'Unnamed: 4', 'Unnamed: 5', 'Rental  Condo  Apts.1', 'Unnamed: 7', 'Apts in the  RMS.1', 'Unnamed: 9', 'Unnamed: 10', 'Rental  Condo  Apts.2', 'Unnamed: 12', 'Apts in the  RMS.2', 'Percentage of  Condo Apts  in Rental', 'Unnamed: 15', 'year']
                            Centre Rental  Condo  Apts Unnamed: 2  \
0                      Halifax CMA                 0.9          a   
1                     Montréal CMA                 2.0          b   
2  Ottawa-Gatineau CMA (Qué. part)                  **        NaN   
3                       Québec CMA                 3.2          d   
4                     Hamilton CMA                 0.3          a   

   Apts in the  RMS Unnamed: 4 Unnamed: 5 Rental  Condo  Apts.1 Unnamed: 7  \
0               1.9          a          Δ                 1,666          c   
1               2.7          a          Δ                 1,277          a   
2               1.6          a        Na

### Part 3:
- Create clean data frame: new column headers, no unneccessary columns, no rows for centre data that are null, metadata or annotations, or a "Total" calculation

In [5]:
# Create a clean data frame with new column headers
cmhc_clean = cmhc_raw.rename(columns={
    'Centre': 'centre',
    'Rental  Condo  Apts': 'vacancy_rental_condo',
    'Apts in the  RMS': 'vacancy_rms',
    'Rental  Condo  Apts.1': 'rent_rental_condo',
    'Apts in the  RMS.1': 'rent_rms',
    'Rental  Condo  Apts.2': 'universe_rental_condo',
    'Apts in the  RMS.2': 'universe_rms',
    'Percentage of  Condo Apts  in Rental': 'pct_condo_in_rental'
})

# Drop unnamed columns
cmhc_clean = cmhc_clean.drop(columns=[col for col in cmhc_clean.columns if "Unnamed" in col])

# Remove rows with metadata characters or annotations in 'centre'
metadata_pattern = r'Δ|©|§|Other|Quality|Source|a — Excellent|b — Very Good|c — Good|d — Poor|—|–|\*\*|\+\+'
cmhc_clean = cmhc_clean[~cmhc_clean["centre"].str.contains(metadata_pattern, regex=True, na=False)].copy()

# Ensure centre value isn't Null or any of the 'Total' rows
cmhc_clean = cmhc_clean[cmhc_clean['centre'].notna() & (cmhc_clean['centre'] != 'Total')]

print(cmhc_clean.head())

                            centre vacancy_rental_condo  vacancy_rms  \
0                      Halifax CMA                  0.9          1.9   
1                     Montréal CMA                  2.0          2.7   
2  Ottawa-Gatineau CMA (Qué. part)                   **          1.6   
3                       Québec CMA                  3.2          2.7   
4                     Hamilton CMA                  0.3          3.5   

  rent_rental_condo rent_rms universe_rental_condo universe_rms  \
0             1,666    1,255                 2,005       52,244   
1             1,277      903                40,354      602,897   
2             1,206      950                 3,501       25,284   
3             1,093      899                 5,754       95,096   
4             1,947    1,291                 3,869       44,258   

   pct_condo_in_rental  year  
0                 25.5  2020  
1                 19.5  2020  
2                 33.9  2020  
3                 16.2  2020  
4        

### Part 4:
- Format numeric column data

In [6]:
# Keep all columns except 'centre' and 'year' for numeric cleaning
num_cols = [col for col in cmhc_clean.columns if col not in ['centre', 'year']]

# Remove non-numeric symbols and commas, then convert to float
for col in num_cols:
    cmhc_clean[col] = (
        cmhc_clean[col].astype(str)
        .str.replace(r'[^0-9.]', '', regex=True)   # keep digits + decimal
        .pipe(pd.to_numeric, errors='coerce')      # convert to float, NaN if fail
    )

## Step 4: Handle remaining missing values using interpolation
- Check for remaining NaN values
- Choose how to handle missing values: interpolation
- Forward/backward fill for remaining missing values in dataframe cmhc_filled
- Identify any remaining centres with NaNs

### Part 1:
- Check for remaining NaN values

In [7]:
print(cmhc_clean.isna().sum())           # check how many NaN values each column has
print(cmhc_clean['year'].value_counts()) # rows per year
print(cmhc_clean.info())                 # confirm dtypes

centre                    0
vacancy_rental_condo      4
vacancy_rms               0
rent_rental_condo        12
rent_rms                  0
universe_rental_condo     0
universe_rms              0
pct_condo_in_rental       0
year                      0
dtype: int64
year
2020    17
2021    17
2022    17
2023    17
2024    17
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 85 entries, 0 to 133
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   centre                 85 non-null     object 
 1   vacancy_rental_condo   81 non-null     float64
 2   vacancy_rms            85 non-null     float64
 3   rent_rental_condo      73 non-null     float64
 4   rent_rms               85 non-null     int64  
 5   universe_rental_condo  85 non-null     int64  
 6   universe_rms           85 non-null     int64  
 7   pct_condo_in_rental    85 non-null     float64
 8   year                   85 

### Part 2:
- Choose how to handle missing values: interpolation

In [8]:
cmhc_clean[num_cols] = (
    cmhc_clean.groupby("centre", group_keys=False)[num_cols]
    .apply(lambda g: g.interpolate(method="linear", limit_direction="both"))
)

### Part 3:
- Forward/backward fill for remaining missing values in dataframe cmhc_filled

In [9]:
cmhc_clean[num_cols] = cmhc_clean.groupby("centre")[num_cols].ffill().bfill()

### Part 4:
- Identify any remaining centres with NaNs

In [10]:
# Sort by centre and year to ensure chronological order
cmhc_clean = cmhc_clean.sort_values(["centre", "year"]).reset_index(drop=True)
# Inspect first 10 values
print(cmhc_clean.head(10))
# Check NaN counts one last time
print(cmhc_clean.isna().sum())

         centre  vacancy_rental_condo  vacancy_rms  rent_rental_condo  \
0   Calgary CMA                   4.9          6.6             1596.0   
1   Calgary CMA                   4.2          5.1             1524.0   
2   Calgary CMA                   1.8          2.7             1648.0   
3   Calgary CMA                   1.0          1.4             1819.0   
4   Calgary CMA                   1.3          4.8             1970.0   
5  Edmonton CMA                   4.0          7.2             1372.0   
6  Edmonton CMA                   5.3          7.3             1412.0   
7  Edmonton CMA                   4.1          4.3             1426.0   
8  Edmonton CMA                   2.5          2.4             1359.0   
9  Edmonton CMA                   2.1          3.1             1466.0   

   rent_rms  universe_rental_condo  universe_rms  pct_condo_in_rental  year  
0      1323                  26520         41995                 37.2  2020  
1      1355                  28384      

## Step 5:
- Export data to .csv

In [11]:
# Final verification before export
assert cmhc_clean.isna().sum().sum() == 0

# Save cleaned CMHC data
cmhc_clean.to_csv("../data/processed/cmhc_clean_2020_2024.csv", index=False)

print("CMHC dataset saved successfully.")

CMHC dataset saved successfully.
